In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:

# Load the CSV file
data_path = os.path.join(path, 'Q1_data.csv')
df_data = pd.read_csv(data_path)

print(f"Shape: {df_data.shape}")


In [ ]:
# Task 2: Write your code here:

df_data.head()

In [ ]:
# Task 3: Write your code here:

# Check data info
df_data.info()

In [ ]:
# Task 4: Write your code here:

df_data.describe()

In [ ]:
# Task 5: Write your code here:

# 1. What does our target variable look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

# Define columns
cols = ['Order_ID']

# Drop
df_clean = df_data.drop(columns=cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")
df_clean.copy().head()


In [ ]:
# Task 2: Write your code here:

# Missing values
print("Missing values:")
print(df_data.isnull().sum())

In [ ]:
# Define stat columns
stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']

# Drop rows with missing stat values
df_clean = df_data.dropna(subset=stat_cols).copy()
print(f"Shape after cleaning: {df_clean.shape}")

# Fill missing delivery time
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna('none')

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) ##inplace changes original data
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_data)

In [ ]:
# Task 4: Write your code here:

categorical_cols = df_data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_data[col] = le.fit_transform(df_data[col])
  label_encoders[col] = le

df_data


#onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
#X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))

In [ ]:
# Task 5: Write your code here:

numerical_cols = df_data.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET


scaler = StandardScaler()
df_data[numerical_cols] = scaler.fit_transform(df_data[numerical_cols])
df_data.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:

X = df_data.drop("Delivery_Time",axis=1)
y = df_data["Delivery_Time"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

In [ ]:
model= RandomForestRegressor(n_estimators=200)

In [ ]:
# Task 2,3,4,5: Write your code here:

# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

In [ ]:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mse = mean_absolute_error(y_test, y_pred)




In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean()) ##same size as 1

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)


print(f"Baseline MAE (using mean target): {baseline_mse:.4f}")


In [ ]:
# Task 1: Write your code here:

coeffs = {}

coeffs['Lasso'] = model['LASSO Regression'].coef_
coeffs['Ridge'] = model['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Plot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task Bonus: Write your code here: